In [0]:
import pandas as pd
import numpy as np
import zipfile
import io

DATA_PATH = '' # Path to the CSV file
TOLERANCE = 0.01  # floating point tolerance

# --- Discover all forecast zip files ---
all_zip_files = [f.path for f in dbutils.fs.ls("dbfs:/FileStore/")
                 if f.name.startswith("forecasts") and f.name.endswith(".zip")]
all_zip_files = sorted(all_zip_files)
print(f"Found {len(all_zip_files)} forecast zip files:")
for zp in all_zip_files:
    print(f"  - {zp}")

# --- Load hierarchy mapping ---
data_map = pd.read_csv(DATA_PATH, usecols=['material', 'customer', 'location', 'CBF', 'PH1', 'PH2', 'PH3']).drop_duplicates()
data_map['ts_id'] = data_map['material'] + '_' + data_map['customer'] + '_' + data_map['location']

# Structural hierarchy: base ts_id -> group key
hierarchy_config = {
    'structural__customer': 'customer',
    'structural__location': 'location',
    'structural__CBF': 'CBF',
    'structural__PH1': 'PH1',
    'structural__PH2': 'PH2',
    'structural__PH3': 'PH3',
    'structural__total': 'total',
}

ts_to_group = {}
for level, col_name in hierarchy_config.items():
    if col_name == 'total':
        ts_to_group[level] = dict.fromkeys(data_map['ts_id'], 'total')
    else:
        ts_to_group[level] = dict(zip(data_map['ts_id'], data_map[col_name]))

# Temporal level pairs for CT checks
temporal_level_pairs = {
    'base': 'temporal__quarter__base',
    'structural__customer': 'temporal__quarter__structural__customer',
    'structural__location': 'temporal__quarter__structural__location',
    'structural__CBF': 'temporal__quarter__structural__CBF',
    'structural__PH1': 'temporal__quarter__structural__PH1',
    'structural__PH2': 'temporal__quarter__structural__PH2',
    'structural__PH3': 'temporal__quarter__structural__PH3',
    'structural__total': 'temporal__quarter__structural__total',
}

def month_to_quarter_start(date_str):
    """Map monthly date to quarter start date."""
    dt = pd.Timestamp(date_str)
    q_start = pd.Timestamp(year=dt.year, month=((dt.month - 1) // 3) * 3 + 1, day=1)
    return str(q_start.date())

print(f"\nHierarchy levels: {list(hierarchy_config.keys())}")
print(f"Base ts_ids: {len(data_map['ts_id'].unique())}")

Found 1 forecast zip files:
  - dbfs:/FileStore/forecasts_regular.zip

Hierarchy levels: ['structural__customer', 'structural__location', 'structural__CBF', 'structural__PH1', 'structural__PH2', 'structural__PH3', 'structural__total']
Base ts_ids: 20000


In [0]:
def validate_zip(zip_path):
    """
    Validate a single forecast zip for structural (CSM) and temporal (CT) coherence.
    Only checks hierarchy levels that actually exist in the zip (subset-aware).
    Returns a DataFrame of results.
    """
    zip_name = zip_path.split('/')[-1]
    print(f"\n{'=' * 70}")
    print(f" {zip_name}")
    print('=' * 70)
    
    # Load zip
    zip_content = spark.read.format("binaryFile").load(zip_path).collect()[0]["content"]
    z = zipfile.ZipFile(io.BytesIO(zip_content))
    file_list = z.namelist()
    print(f"  Size: {len(zip_content)/1024/1024:.1f} MB, {len(file_list)} files")
    
    # Detect root folder prefix inside the zip
    root_prefix = next((f.split('/')[0] for f in file_list if '/' in f and not f.startswith('__')), 'forecasts')
    
    # Discover which structural levels actually exist in this zip
    available_structural = sorted(set(
        '/'.join(f.split('/')[1:-1])
        for f in file_list
        if f.endswith('.csv') and 'structural__' in f and 'temporal__' not in f
    ))
    # Discover which temporal quarter levels exist
    available_temporal = sorted(set(
        '/'.join(f.split('/')[1:-1])
        for f in file_list
        if f.endswith('.csv') and 'temporal__quarter__' in f
    ))
    
    # Determine what's missing vs full hierarchy
    all_structural = sorted(hierarchy_config.keys())
    all_temporal = sorted(temporal_level_pairs.values())
    missing_structural = sorted(set(all_structural) - set(available_structural))
    missing_temporal = sorted(set(all_temporal) - set(available_temporal))
    
    print(f"  Structural levels: {len(available_structural)}/{len(all_structural)} available")
    if missing_structural:
        print(f"    Missing: {missing_structural}")
    print(f"  Temporal levels:   {len(available_temporal)}/{len(all_temporal)} available")
    if missing_temporal:
        print(f"    Missing: {missing_temporal}")
    
    # Discover reconciled methods
    recon_methods = sorted(set(
        n.split('/')[-1].replace('_forecasts.csv', '')
        for n in file_list
        if '/base/' in n and ('__ct' in n or '__csm' in n)
    ))
    csm_methods = [m for m in recon_methods if '__csm' in m]
    ct_methods = [m for m in recon_methods if '__ct' in m]
    print(f"  Methods: {len(recon_methods)} ({len(csm_methods)} CSM, {len(ct_methods)} CT)")
    
    results = []
    
    # --- STRUCTURAL COHERENCE (CSM) ---
    # Only check levels that exist in this zip
    zip_hierarchy = {k: v for k, v in hierarchy_config.items() if k in available_structural}
    
    for method in csm_methods:
        base_file = f'{root_prefix}/base/{method}_forecasts.csv'
        if base_file not in file_list:
            continue
        base_fc = pd.read_csv(z.open(base_file))
        
        for level, col_name in zip_hierarchy.items():
            higher_file = f'{root_prefix}/{level}/{method}_forecasts.csv'
            if higher_file not in file_list:
                continue  # skip — level folder exists but not this method
            
            base_fc_copy = base_fc.copy()
            base_fc_copy['group_key'] = base_fc_copy['ts_id'].map(ts_to_group[level])
            base_summed = base_fc_copy.groupby(['date', 'group_key'])['forecast'].sum().reset_index()
            base_summed.rename(columns={'forecast': 'base_sum', 'group_key': 'ts_id'}, inplace=True)
            
            higher_fc = pd.read_csv(z.open(higher_file))
            merged = pd.merge(base_summed, higher_fc, on=['date', 'ts_id'])
            
            if len(merged) == 0:
                results.append({'check_type': 'structural (CSM)', 'method': method,
                               'level': level, 'status': '✗ NO MATCH',
                               'max_abs_diff': np.nan, 'n_checks': 0})
                continue
            
            merged['abs_diff'] = (merged['base_sum'] - merged['forecast']).abs()
            max_abs = merged['abs_diff'].max()
            is_coherent = max_abs < TOLERANCE
            
            results.append({
                'check_type': 'structural (CSM)', 'method': method, 'level': level,
                'status': '✓' if is_coherent else '✗',
                'max_abs_diff': max_abs, 'n_checks': len(merged)
            })
    
    # --- TEMPORAL COHERENCE (CT) ---
    # Only check level pairs where BOTH monthly and quarterly exist
    zip_temporal_pairs = {
        k: v for k, v in temporal_level_pairs.items()
        if k in (['base'] + available_structural) and v in available_temporal
    }
    
    for method in ct_methods:
        for monthly_level, quarterly_level in zip_temporal_pairs.items():
            monthly_file = f'{root_prefix}/{monthly_level}/{method}_forecasts.csv'
            quarterly_file = f'{root_prefix}/{quarterly_level}/{method}_forecasts.csv'
            
            if monthly_file not in file_list or quarterly_file not in file_list:
                continue  # skip — level pair not fully available for this method
            
            monthly_fc = pd.read_csv(z.open(monthly_file))
            monthly_fc['quarter'] = monthly_fc['date'].apply(month_to_quarter_start)
            monthly_summed = monthly_fc.groupby(['quarter', 'ts_id'])['forecast'].sum().reset_index()
            monthly_summed.rename(columns={'quarter': 'date', 'forecast': 'monthly_sum'}, inplace=True)
            
            quarterly_fc = pd.read_csv(z.open(quarterly_file))
            merged = pd.merge(monthly_summed, quarterly_fc, on=['date', 'ts_id'])
            
            if len(merged) == 0:
                results.append({'check_type': 'temporal (CT)', 'method': method,
                               'level': f'{monthly_level} -> quarter',
                               'status': '✗ NO MATCH', 'max_abs_diff': np.nan, 'n_checks': 0})
                continue
            
            merged['abs_diff'] = (merged['monthly_sum'] - merged['forecast']).abs()
            max_abs = merged['abs_diff'].max()
            is_coherent = max_abs < TOLERANCE
            
            results.append({
                'check_type': 'temporal (CT)', 'method': method,
                'level': f'{monthly_level} -> quarter',
                'status': '✓' if is_coherent else '✗',
                'max_abs_diff': max_abs, 'n_checks': len(merged)
            })
    
    df = pd.DataFrame(results)
    passed = (df['status'] == '✓').sum() if len(df) > 0 else 0
    total = len(df)
    struct_df = df[df['check_type'] == 'structural (CSM)']
    temp_df = df[df['check_type'] == 'temporal (CT)']
    sp = (struct_df['status'] == '✓').sum() if len(struct_df) > 0 else 0
    tp = (temp_df['status'] == '✓').sum() if len(temp_df) > 0 else 0
    
    print(f"\n  RESULTS:")
    print(f"    Structural (CSM): {sp}/{len(struct_df)} passed ({len(zip_hierarchy)} levels checked)")
    print(f"    Temporal (CT):    {tp}/{len(temp_df)} passed ({len(zip_temporal_pairs)} level pairs checked)")
    print(f"    {'—' * 40}")
    
    if passed == total and total > 0:
        print(f"    ✓ ALL {total} CHECKS PASSED")
    elif total == 0:
        print(f"    ⚠️ No reconciled methods found")
    else:
        print(f"    ✗ {total - passed} FAILURES out of {total} checks")
        display(df[df['status'] != '✓'])
    
    # --- Build per-model file availability matrix ---
    # All levels to check (base + structural + temporal)
    all_levels_in_zip = ['base'] + available_structural + available_temporal
    
    # Get all methods present in this zip (not just reconciled)
    all_methods_in_zip = sorted(set(
        n.split('/')[-1].replace('_forecasts.csv', '')
        for n in file_list
        if n.endswith('_forecasts.csv')
    ))
    
    # Build availability dict: {method: {level: True/False}}
    file_availability = {}
    for method in all_methods_in_zip:
        file_availability[method] = {}
        for level in all_levels_in_zip:
            fpath = f'{root_prefix}/{level}/{method}_forecasts.csv'
            file_availability[method][level] = fpath in file_list
    
    df['zip'] = zip_name
    return df, {
        'zip': zip_name,
        'root_prefix': root_prefix,
        'available_structural': available_structural,
        'available_temporal': available_temporal,
        'missing_structural': missing_structural,
        'missing_temporal': missing_temporal,
        'csm_methods': csm_methods,
        'ct_methods': ct_methods,
        'all_methods': all_methods_in_zip,
        'all_levels': all_levels_in_zip,
        'file_availability': file_availability,
    }

In [0]:
# Run validation for each zip independently
raw_results = [validate_zip(zp) for zp in all_zip_files]
all_dfs = [r[0] for r in raw_results]
all_meta = [r[1] for r in raw_results]
results_df = pd.concat(all_dfs, ignore_index=True)

# --- Grand coherence summary ---
print("\n" + "=" * 70)
print(" GRAND COHERENCE SUMMARY")
print("=" * 70)
total = len(results_df)
passed = (results_df['status'] == '✓').sum()
print(f"  Total checks across all zips: {passed}/{total} passed")
if passed == total:
    print("\n   All reconciled forecasts are perfectly coherent!")
    print("     - CSM: sum(base) == higher structural level ✓")
    print("     - CT: sum(monthly) == quarterly ✓")
else:
    print(f"\n   {total - passed} failures detected (see per-zip details above)")

# --- Per-zip expected scope ---
# Define what IS expected per zip type so we only flag genuine gaps.
# - "noneg" zips: MinT_shrink and WLS_var methods are NOT conducted
# - "subset" zips: PH2, PH3, customer levels are expected to be absent

def get_expected_scope(zip_name):
    """Return (method_expected_fn, level_expected_fn) for a zip."""
    is_noneg = 'noneg' in zip_name
    is_subset = 'subset' in zip_name
    
    def method_expected(m):
        if is_noneg and ('MinT_shrink' in m or 'WLS_var' in m):
            return False
        return True
    
    subset_excluded = {'structural__PH2', 'structural__PH3', 'structural__customer',
                       'temporal__quarter__structural__PH2',
                       'temporal__quarter__structural__PH3',
                       'temporal__quarter__structural__customer'}
    def level_expected(lvl):
        if is_subset and lvl in subset_excluded:
            return False
        return True
    
    return method_expected, level_expected

# --- Cross-zip missing file comparison (scope-aware) ---
# Use the most complete zip (forecasts_reg.zip) as the reference
ref_meta = next((m for m in all_meta if 'forecasts_reg.zip' in m['zip']), all_meta[0])
ref_name = ref_meta['zip']

print("\n" + "=" * 70)
print(f" UNEXPECTED MISSING FILES (reference: {ref_name})")
print("=" * 70)
print(f"  Reference has: {len(ref_meta['available_structural'])} structural, "
      f"{len(ref_meta['available_temporal'])} temporal, "
      f"{len(ref_meta['csm_methods'])} CSM methods, {len(ref_meta['ct_methods'])} CT methods")

for meta in all_meta:
    if meta['zip'] == ref_name:
        continue
    
    method_expected, level_expected = get_expected_scope(meta['zip'])
    
    print(f"\n  {meta['zip']}:")
    
    # Only flag levels missing that ARE expected
    unexpected_struct = [l for l in meta['missing_structural'] if level_expected(l)]
    unexpected_temp = [l for l in meta['missing_temporal'] if level_expected(l)]
    
    if unexpected_struct:
        print(f"    ✗ Unexpected missing structural levels ({len(unexpected_struct)}):")
        for lvl in unexpected_struct:
            print(f"      - {lvl}")
    else:
        print(f"    Structural levels: ✓")
    
    if unexpected_temp:
        print(f"    ✗ Unexpected missing temporal levels ({len(unexpected_temp)}):")
        for lvl in unexpected_temp:
            print(f"      - {lvl}")
    else:
        print(f"    Temporal levels: ✓")
    
    # Only flag methods missing that ARE expected
    missing_csm = sorted(m for m in set(ref_meta['csm_methods']) - set(meta['csm_methods'])
                         if method_expected(m))
    missing_ct = sorted(m for m in set(ref_meta['ct_methods']) - set(meta['ct_methods'])
                        if method_expected(m))
    if missing_csm:
        print(f"    ✗ Unexpected missing CSM methods ({len(missing_csm)}):")
        for m in missing_csm:
            print(f"      - {m}")
    else:
        print(f"    CSM methods: ✓")
    if missing_ct:
        print(f"    ✗ Unexpected missing CT methods ({len(missing_ct)}):")
        for m in missing_ct:
            print(f"      - {m}")
    else:
        print(f"    CT methods: ✓")

# --- Per-model file availability (scope-aware, unexpected gaps only) ---

def short_level(level):
    """Shorten level name for display."""
    return (level
            .replace('structural__', '')
            .replace('temporal__quarter__structural__', 'quarter_')
            .replace('temporal__quarter__', 'quarter_'))

print("\n" + "=" * 70)
print(" PER-MODEL FILE AVAILABILITY (unexpected gaps only)")
print("=" * 70)

for meta in all_meta:
    zip_name = meta['zip']
    avail = meta['file_availability']
    csm_methods = meta['csm_methods']
    ct_methods = meta['ct_methods']
    available_structural = meta['available_structural']
    available_temporal = meta['available_temporal']
    
    method_expected, level_expected = get_expected_scope(zip_name)
    
    print(f"\n  {'-' * 60}")
    print(f"  {zip_name}")
    print(f"  {'-' * 60}")
    
    gaps = []  # collect (type, level_short, missing_models)
    
    # CSM: check only expected methods at expected levels
    expected_csm = [m for m in csm_methods if method_expected(m)]
    csm_expected_levels = ['base'] + [l for l in available_structural if level_expected(l)]
    
    for level in csm_expected_levels:
        present = [m for m in expected_csm if avail.get(m, {}).get(level, False)]
        missing = [m for m in expected_csm if not avail.get(m, {}).get(level, False)]
        if missing:
            missing_models = sorted(set(m.split('__')[0] for m in missing))
            gaps.append(('CSM', short_level(level), missing_models))
    
    # CT: check only expected methods at expected levels
    expected_ct = [m for m in ct_methods if method_expected(m)]
    ct_expected_levels = (['base'] + [l for l in available_structural if level_expected(l)]
                          + [l for l in available_temporal if level_expected(l)])
    
    for level in ct_expected_levels:
        present = [m for m in expected_ct if avail.get(m, {}).get(level, False)]
        missing = [m for m in expected_ct if not avail.get(m, {}).get(level, False)]
        if missing:
            missing_models = sorted(set(m.split('__')[0] for m in missing))
            gaps.append(('CT', short_level(level), missing_models))
    
    if not gaps:
        print(f"    All expected models present at all expected levels ✓")
    else:
        for check_type, level_short, missing_models in gaps:
            print(f"    {check_type} {level_short}: missing models: {', '.join(missing_models)}")


 forecasts_csm_subset.zip
  Size: 164.0 MB, 781 files
  Structural levels: 4/7 available
    Missing: ['structural__PH2', 'structural__PH3', 'structural__customer']
  Temporal levels:   5/8 available
    Missing: ['temporal__quarter__structural__PH2', 'temporal__quarter__structural__PH3', 'temporal__quarter__structural__customer']
  Methods: 88 (44 CSM, 44 CT)

  RESULTS:
    Structural (CSM): 176/176 passed (4 levels checked)
    Temporal (CT):    220/220 passed (5 level pairs checked)
    ————————————————————————————————————————
    ✓ ALL 396 CHECKS PASSED

 forecasts_csm_subset_noneg.zip
  Size: 86.1 MB, 411 files
  Structural levels: 4/7 available
    Missing: ['structural__PH2', 'structural__PH3', 'structural__customer']
  Temporal levels:   5/8 available
    Missing: ['temporal__quarter__structural__PH2', 'temporal__quarter__structural__PH3', 'temporal__quarter__structural__customer']
  Methods: 44 (22 CSM, 22 CT)

  RESULTS:
    Structural (CSM): 88/88 passed (4 levels checked)

In [0]:
def compute_clip_discrepancy(zip_path):
    """
    Load an unclipped forecast zip, manually clip negatives to 0,
    and measure how much clipping breaks hierarchical (structural) consistency.
    
    Compares:
      - Original: sum(base) vs higher_level (should be coherent)
      - Clipped:  sum(clip(base, 0)) vs clip(higher_level, 0) (will diverge)
    
    Returns DataFrame with per-method/level/date/group discrepancy.
    """
    zip_name = zip_path.split('/')[-1]
    print(f"\nProcessing: {zip_name}")
    
    zip_content = spark.read.format("binaryFile").load(zip_path).collect()[0]["content"]
    z = zipfile.ZipFile(io.BytesIO(zip_content))
    file_list = z.namelist()
    
    root_prefix = next((f.split('/')[0] for f in file_list if '/' in f and not f.startswith('__')), 'forecasts')
    
    # Discover structural levels and CSM methods
    available_structural = sorted(set(
        '/'.join(f.split('/')[1:-1])
        for f in file_list
        if f.endswith('.csv') and 'structural__' in f and 'temporal__' not in f
    ))
    csm_methods = sorted(set(
        n.split('/')[-1].replace('_forecasts.csv', '')
        for n in file_list
        if '/base/' in n and '__csm' in n
    ))
    
    zip_hierarchy = {k: v for k, v in hierarchy_config.items() if k in available_structural}
    
    print(f"  Methods: {len(csm_methods)}, Structural levels: {len(zip_hierarchy)}")
    
    rows = []
    for method in csm_methods:
        base_file = f'{root_prefix}/base/{method}_forecasts.csv'
        if base_file not in file_list:
            continue
        base_fc = pd.read_csv(z.open(base_file))
        
        for level, col_name in zip_hierarchy.items():
            higher_file = f'{root_prefix}/{level}/{method}_forecasts.csv'
            if higher_file not in file_list:
                continue
            
            higher_fc = pd.read_csv(z.open(higher_file))
            
            # --- Original (unclipped) aggregation ---
            base_fc_copy = base_fc.copy()
            base_fc_copy['group_key'] = base_fc_copy['ts_id'].map(ts_to_group[level])
            base_summed = base_fc_copy.groupby(['date', 'group_key'])['forecast'].sum().reset_index()
            base_summed.rename(columns={'forecast': 'orig_base_sum', 'group_key': 'ts_id'}, inplace=True)
            
            # --- Clipped aggregation: clip base to 0, then sum ---
            base_fc_clip = base_fc.copy()
            base_fc_clip['forecast'] = base_fc_clip['forecast'].clip(lower=0)
            base_fc_clip['group_key'] = base_fc_clip['ts_id'].map(ts_to_group[level])
            clip_base_summed = base_fc_clip.groupby(['date', 'group_key'])['forecast'].sum().reset_index()
            clip_base_summed.rename(columns={'forecast': 'clip_base_sum', 'group_key': 'ts_id'}, inplace=True)
            
            # --- Clipped higher level ---
            higher_fc_clip = higher_fc.copy()
            higher_fc_clip['clip_higher'] = higher_fc_clip['forecast'].clip(lower=0)
            
            # Merge all together
            merged = base_summed.merge(clip_base_summed, on=['date', 'ts_id'])
            merged = merged.merge(
                higher_fc[['date', 'ts_id', 'forecast']].rename(columns={'forecast': 'orig_higher'}),
                on=['date', 'ts_id']
            )
            merged = merged.merge(
                higher_fc_clip[['date', 'ts_id', 'clip_higher']],
                on=['date', 'ts_id']
            )
            
            if len(merged) == 0:
                continue
            
            # Original discrepancy (should be ~0 for reconciled forecasts)
            merged['orig_abs_diff'] = (merged['orig_base_sum'] - merged['orig_higher']).abs()
            
            # Clipped discrepancy: sum(clip(base)) vs clip(higher)
            merged['clip_abs_diff'] = (merged['clip_base_sum'] - merged['clip_higher']).abs()
            
            # How many base series were negative (got clipped)?
            neg_count = (base_fc_copy.assign(is_neg=base_fc_copy['forecast'] < 0)
             .groupby(['date', 'group_key'])['is_neg'].sum().reset_index())
            neg_count.rename(columns={'is_neg': 'n_clipped_series', 'group_key': 'ts_id'}, inplace=True)
            merged = merged.merge(neg_count, on=['date', 'ts_id'], how='left')
            merged['n_clipped_series'] = merged['n_clipped_series'].fillna(0).astype(int)
            
            merged['method'] = method
            merged['level'] = level
            rows.append(merged[['method', 'level', 'date', 'ts_id',
                                'orig_base_sum', 'orig_higher', 'orig_abs_diff',
                                'clip_base_sum', 'clip_higher', 'clip_abs_diff',
                                'n_clipped_series']])
    
    if not rows:
        return pd.DataFrame()
    
    df = pd.concat(rows, ignore_index=True)
    df['zip'] = zip_name
    return df


# --- Run on forecasts_regular.zip (unclipped, then manually clip to 0) ---
print("HIERARCHICAL CONSISTENCY: IMPACT OF CLIPPING NEGATIVES TO ZERO")
print("=" * 70)
print("Source: forecasts_regular.zip (unclipped reconciled forecasts)")
print("Logic: compare sum(clip(base, 0)) vs clip(higher_level, 0)")
print("       Reconciled forecasts satisfy sum(base) == higher_level,")
print("       but clipping breaks this: sum(clip(base)) != clip(higher).")
print("=" * 70)

reg_zip = next((z for z in all_zip_files if 'forecasts_regular.zip' in z), None)
if reg_zip is None:
    raise FileNotFoundError("forecasts_regular.zip not found in dbfs:/FileStore/")

all_discrepancy = compute_clip_discrepancy(reg_zip)

# --- Summary ---
print("\n" + "=" * 70)
print(" SUMMARY: forecasts_regular.zip — clipping discrepancy")
print("=" * 70)

agg_pct = all_discrepancy['clip_abs_diff'].sum() / all_discrepancy['clip_higher'].sum() * 100

print(f"  Total comparison points:     {len(all_discrepancy):,}")
print(f"  Points with clipping impact:  {(all_discrepancy['clip_abs_diff'] > TOLERANCE).sum():,} "
      f"({(all_discrepancy['clip_abs_diff'] > TOLERANCE).mean() * 100:.1f}%)")
print(f"  Mean absolute discrepancy:    {all_discrepancy['clip_abs_diff'].mean():.2f}")
print(f"  Max absolute discrepancy:     {all_discrepancy['clip_abs_diff'].max():.2f}")
print(f"  Aggregate % discrepancy:      {agg_pct:.2f}%  (sum(abs_diff) / sum(clip_higher))")
print(f"  Groups with negative series:  {(all_discrepancy['n_clipped_series'] > 0).sum():,} "
      f"({(all_discrepancy['n_clipped_series'] > 0).mean() * 100:.1f}%)")

# --- Breakdown by hierarchy level ---
print("\n" + "-" * 70)
print(" DISCREPANCY BY HIERARCHY LEVEL")
print("-" * 70)

level_summary = all_discrepancy.groupby('level').agg(
    mean_clip_diff=('clip_abs_diff', 'mean'),
    max_clip_diff=('clip_abs_diff', 'max'),
    total_clip_diff=('clip_abs_diff', 'sum'),
    total_clip_higher=('clip_higher', 'sum'),
    n_impacted=('clip_abs_diff', lambda x: (x > TOLERANCE).sum()),
    n_points=('clip_abs_diff', 'count'),
    mean_neg_series=('n_clipped_series', 'mean')
).reset_index()
level_summary['agg_pct'] = level_summary['total_clip_diff'] / level_summary['total_clip_higher'] * 100
level_summary['pct_impacted'] = level_summary['n_impacted'] / level_summary['n_points'] * 100
level_summary = level_summary.sort_values('total_clip_diff', ascending=False)
display(level_summary[['level', 'mean_clip_diff', 'max_clip_diff', 'total_clip_diff', 'agg_pct', 'n_impacted', 'n_points', 'pct_impacted', 'mean_neg_series']])

# --- Breakdown by method ---
print("\n" + "-" * 70)
print(" DISCREPANCY BY RECONCILIATION METHOD")
print("-" * 70)

method_summary = all_discrepancy.groupby('method').agg(
    mean_clip_diff=('clip_abs_diff', 'mean'),
    max_clip_diff=('clip_abs_diff', 'max'),
    total_clip_diff=('clip_abs_diff', 'sum'),
    total_clip_higher=('clip_higher', 'sum'),
    n_impacted=('clip_abs_diff', lambda x: (x > TOLERANCE).sum()),
    n_points=('clip_abs_diff', 'count'),
    mean_neg_series=('n_clipped_series', 'mean')
).reset_index()
method_summary['agg_pct'] = method_summary['total_clip_diff'] / method_summary['total_clip_higher'] * 100
method_summary['pct_impacted'] = method_summary['n_impacted'] / method_summary['n_points'] * 100
method_summary = method_summary.sort_values('total_clip_diff', ascending=False)
display(method_summary[['method', 'mean_clip_diff', 'max_clip_diff', 'total_clip_diff', 'agg_pct', 'n_impacted', 'n_points', 'pct_impacted', 'mean_neg_series']])

HIERARCHICAL CONSISTENCY: IMPACT OF CLIPPING NEGATIVES TO ZERO
Source: forecasts_regular.zip (unclipped reconciled forecasts)
Logic: compare sum(clip(base, 0)) vs clip(higher_level, 0)
       Reconciled forecasts satisfy sum(base) == higher_level,
       but clipping breaks this: sum(clip(base)) != clip(higher).

Processing: forecasts_regular.zip
  Methods: 66, Structural levels: 7

 SUMMARY: forecasts_regular.zip — clipping discrepancy
  Total comparison points:     535,392
  Points with clipping impact:  148,146 (27.7%)
  Mean absolute discrepancy:    1121.85
  Max absolute discrepancy:     6144882.47
  Aggregate % discrepancy:      6.00%  (sum(abs_diff) / sum(clip_higher))
  Groups with negative series:  164,814 (30.8%)

----------------------------------------------------------------------
 DISCREPANCY BY HIERARCHY LEVEL
----------------------------------------------------------------------


level,mean_clip_diff,max_clip_diff,total_clip_diff,agg_pct,n_impacted,n_points,pct_impacted,mean_neg_series
structural__total,247767.48563219828,6144882.473228533,9.811592431035052E7,6.9200018561741645,264,396,66.66666666666666,2260.15404040404
structural__CBF,15236.068970042019,1559718.586556457,9.653573299418624E7,6.800973183050045,3018,6336,47.63257575757576,141.2596275252525
structural__location,23517.202620528413,2140851.9900136604,9.312812237729251E7,6.545193062791077,1926,3960,48.63636363636364,226.01540404040404
structural__PH1,1911.6953005024204,1559718.586556457,8.630157264588127E7,6.03645052207206,16280,45144,36.06237816764132,19.82591263512316
structural__customer,1654.9608310824417,821362.4774175722,8.323129011679816E7,5.809220979758407,19162,50292,38.101487314085745,17.79648850711843
structural__PH2,469.66354556423636,718206.7336157578,7.365075856120129E7,5.1063914467166835,42830,156816,27.312264054688296,5.707459697990001
structural__PH3,255.6957301183238,292463.50399451645,6.966379027927709E7,4.816650458090993,64666,272448,23.735171482264505,3.285107616866338



----------------------------------------------------------------------
 DISCREPANCY BY RECONCILIATION METHOD
----------------------------------------------------------------------


method,mean_clip_diff,max_clip_diff,total_clip_diff,agg_pct,n_impacted,n_points,pct_impacted,mean_neg_series
lightgbm__csm_MinT_shrink,23168.621252124965,6144882.473228533,1.879438555972377E8,81.90749193364127,5700,8112,70.26627218934911,35.301158777120314
lightgbm_direct__csm_MinT_shrink,9033.528818269157,2366214.300989792,7.32799857737994E7,54.384855255615115,4993,8112,61.5507889546351,16.485207100591715
random_forest__csm_MinT_shrink,8951.73883291996,2294452.3081794917,7.261650541264671E7,48.31771556315884,5792,8112,71.40039447731755,29.48853550295858
arima__csm_MinT_shrink,3689.896707949149,1788454.7668615743,2.9932442094883498E7,15.708778055088763,5252,8112,64.74358974358975,21.182938856015777
nhits__csm_OLS,3291.428568762421,809930.4381259703,2.670006854980076E7,17.72276711413621,4216,8112,51.972386587771204,36.04585798816568
random_forest_direct__csm_OLS,2996.3527438030205,908222.8042043145,2.4306413457730103E7,16.593236434371757,4271,8112,52.650394477317555,37.67246055226825
deepar__csm_OLS,2863.0081165695215,909990.3312349399,2.322472184161196E7,15.242235567273358,3984,8112,49.112426035502956,31.040064102564102
lightgbm_direct__csm_OLS,2559.2720453021066,682731.7054950609,2.0760814831490688E7,13.910785262520239,3904,8112,48.12623274161736,35.0362426035503
lightgbm__csm_OLS,2485.5307219896604,715129.276557989,2.0162625216780126E7,13.374968978465281,4621,8112,56.964990138067066,31.133259368836292
random_forest__csm_OLS,2318.3904842087536,617732.9566103206,1.880678360790141E7,12.582239940544692,4757,8112,58.641518737672584,37.24445266272189
